# 02_18 Kaggle MONAI DynUNet PHE-only fair baseline

Notebook nay dung official MONAI `DynUNet` de train PHE-only baseline tren PHE-SICH CT.

Muc tieu so sanh voi `02_12`:

```text
input  channel 0 = CT only
target label     = manual PHE mask
model            = MONAI DynUNet 3D
split            = same train/val/test IDs as 02_12
ICH              = not used
epochs           = 250
```

Ghi chu fairness:

- Cong bang ve data, split, epoch budget, full-volume test inference va metrics.
- Khac voi `02_12`: day la custom MONAI training loop, khong phai nnU-Net auto-planner/trainer. Vi vay day la baseline model-family, khong phai replacement 1-1 trong nnU-Net.


In [1]:
from pathlib import Path
from collections import OrderedDict
import os
import sys
import re
import json
import math
import time
import random
import gc
import subprocess

import numpy as np
import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().resolve()
LOCAL_ROOT = Path.cwd().resolve()

# Optional manual override. Neu auto-detect sai, gan path nay tren Kaggle.
# Example:
# USER_PHE_ROOT = Path('/kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT')
USER_PHE_ROOT = None

OUTPUT_ROOT = WORK_ROOT / 'outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline'
PRED_DIR = OUTPUT_ROOT / 'predictions_dynunet_best'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
for p in [OUTPUT_ROOT, PRED_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_SCAN_IDS = {'0013', '0014', '0060', '0078', '0080', '0115', '0119', '0130', '0138', '0160'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0051', '0061', '0084', '0095', '0096', '0113', '0116', '0167'}

# Fair-to-02_12 budget.
MAX_EPOCHS = 250
TRAIN_PATCHES_PER_EPOCH = 256
PATCH_SIZE = (32, 128, 128)  # z, y, x. Divisible by DynUNet stride product.
BATCH_SIZE = 1
FG_PATCH_PROB = 0.85
VALIDATE_EVERY = 10
SAVE_EVERY = 10
NUM_WORKERS = 0
MAX_CACHE_VOLUMES = 10
PATCHES_PER_CASE_BURST = 4  # reuse one loaded case for a few patches to reduce Kaggle input IO

# CT-only preprocessing. Window nay giong cac custom baseline PHE-only gan day.
CT_WINDOW = (-100.0, 200.0)

# DynUNet config. 3D, anisotropic-friendly early stride, residual blocks, deep supervision.
DYNUNET_KERNELS = ((1, 3, 3), (3, 3, 3), (3, 3, 3), (3, 3, 3), (3, 3, 3))
DYNUNET_STRIDES = ((1, 1, 1), (1, 2, 2), (2, 2, 2), (2, 2, 2), (2, 2, 2))
DYNUNET_UPSAMPLE_KERNELS = DYNUNET_STRIDES[1:]
DYNUNET_FILTERS = (32, 64, 128, 256, 320)
DYNUNET_DEEP_SUPERVISION = True
DYNUNET_DEEP_SUPR_NUM = 2
DYNUNET_RES_BLOCK = True

LR = 3e-4
WEIGHT_DECAY = 1e-5
USE_AMP = True
GRAD_CLIP_NORM = 12.0
SLIDING_WINDOW_BATCH_SIZE = 1
SLIDING_WINDOW_OVERLAP = 0.5

print('IS_KAGGLE     :', IS_KAGGLE)
print('WORK_ROOT     :', WORK_ROOT)
print('Output root   :', OUTPUT_ROOT)
print('Model         : MONAI DynUNet')
print('Epochs        :', MAX_EPOCHS)
print('Patch size zyx:', PATCH_SIZE)
print('No ICH input  : True')


IS_KAGGLE     : True
WORK_ROOT     : /kaggle/working
Output root   : /kaggle/working/outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline
Model         : MONAI DynUNet
Epochs        : 250
Patch size zyx: (32, 128, 128)
No ICH input  : True


## 0. Install/import official MONAI DynUNet

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass


def run_cmd(cmd):
    print('>>>', ' '.join(map(str, cmd)))
    t0 = time.time()
    subprocess.check_call(cmd)
    print(f'Done in {(time.time() - t0) / 60:.1f} min')


def ensure_import(import_name, pip_name=None):
    import importlib
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        run_cmd([sys.executable, '-m', 'pip', 'install', pip_name])
        return importlib.import_module(import_name)


seed_everything(SEED)

monai = ensure_import('monai')
sitk = ensure_import('SimpleITK')
ensure_import('scipy')

import SimpleITK as sitk
from scipy import ndimage as ndi

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from monai.networks.nets import DynUNet
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference

print('torch      :', torch.__version__)
print('cuda       :', torch.version.cuda)
print('cuda avail :', torch.cuda.is_available())
print('monai      :', monai.__version__)
print('DynUNet    :', DynUNet)


>>> /usr/bin/python3 -m pip install monai
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.4 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Done in 0.1 min


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-06-18 17:12:13.215284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781802733.625959      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781802733.735329      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781802734.739209      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781802734.739244      23 computation_placer.cc:1

torch      : 2.10.0+cu128
cuda       : 12.8
cuda avail : True
monai      : 1.5.2
DynUNet    : <class 'monai.networks.nets.dynunet.DynUNet'>


## 1. Locate PHE data and build split manifest

In [3]:
def strip_nii_suffix(path_like):
    name = Path(str(path_like)).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return Path(name).stem


def scan_id_from_name(path_like):
    stem = strip_nii_suffix(path_like)
    m = re.search(r'(\d{4})$', stem)
    if not m:
        m = re.search(r'(\d+)$', stem)
    if not m:
        raise ValueError(f'Cannot parse scan id from {path_like}')
    return m.group(1).zfill(4)


def make_case_id(scan_id):
    scan_id = str(scan_id).strip()
    if scan_id.lower().startswith('phe_'):
        return scan_id
    if scan_id.isdigit():
        return f'phe_{int(scan_id):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in scan_id).strip('_')
    return f'phe_{safe}'


def has_nifti_pair_root(path_like):
    p = Path(path_like)
    if not (p / 'set').exists() or not (p / 'label').exists():
        return False
    image_count = len(list((p / 'set').glob('*.nii'))) + len(list((p / 'set').glob('*.nii.gz')))
    mask_count = len(list((p / 'label').glob('*.nii'))) + len(list((p / 'label').glob('*.nii.gz')))
    return image_count > 0 and mask_count > 0


def find_phe_root():
    if USER_PHE_ROOT is not None:
        p = Path(USER_PHE_ROOT)
        if has_nifti_pair_root(p):
            return p
        raise FileNotFoundError(f'USER_PHE_ROOT does not contain set/label NIfTI pairs: {p}')

    candidates = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        LOCAL_ROOT / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([
                base / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'NIFIT',
                base,
            ])
    for p in candidates:
        if has_nifti_pair_root(p):
            return p

    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    found = []
    for root in search_roots:
        for set_dir in root.rglob('set'):
            p = set_dir.parent
            if has_nifti_pair_root(p):
                found.append(p)
    if found:
        found = sorted(found, key=lambda p: (('SubdatasetA_NIFIT' not in str(p)), len(str(p))))
        return found[0]
    raise FileNotFoundError('Could not find PHE root with set/ and label/. Set USER_PHE_ROOT manually.')


def split_from_scan_id(scan_id):
    scan_id = str(scan_id).strip()
    scan_id4 = f'{int(scan_id):04d}' if scan_id.isdigit() else scan_id
    if scan_id4 in TEST_SCAN_IDS:
        return 'test'
    if scan_id4 in VAL_SCAN_IDS:
        return 'val'
    return 'train'


PHE_ROOT = find_phe_root()
IMAGE_DIR = PHE_ROOT / 'set'
MASK_DIR = PHE_ROOT / 'label'
print('PHE root :', PHE_ROOT)
print('Image dir:', IMAGE_DIR)
print('Mask dir :', MASK_DIR)

image_files = sorted(list(IMAGE_DIR.glob('*.nii')) + list(IMAGE_DIR.glob('*.nii.gz')))
mask_files = sorted(list(MASK_DIR.glob('*.nii')) + list(MASK_DIR.glob('*.nii.gz')))
mask_by_scan = {scan_id_from_name(p): p for p in mask_files}

rows = []
missing_masks = []
for image_path in image_files:
    scan_id = scan_id_from_name(image_path)
    mask_path = mask_by_scan.get(scan_id)
    if mask_path is None:
        missing_masks.append((scan_id, str(image_path)))
        continue
    rows.append({
        'scan_id': scan_id,
        'case_id': make_case_id(scan_id),
        'split': split_from_scan_id(scan_id),
        'image_path': str(image_path),
        'label_path': str(mask_path),
    })

if missing_masks:
    print('WARNING: images without matching masks:', len(missing_masks))
    print(missing_masks[:5])

split_df = pd.DataFrame(rows).sort_values(['split', 'scan_id']).reset_index(drop=True)
manifest_csv = OUTPUT_ROOT / '02_18_dynunet_phe_only_manifest.csv'
split_df.to_csv(manifest_csv, index=False)

print('Images:', len(image_files), 'Masks:', len(mask_files), 'Matched:', len(split_df))
display(split_df['split'].value_counts().rename_axis('split').reset_index(name='n'))
print('Manifest:', manifest_csv)
display(split_df.head())


PHE root : /kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT
Image dir: /kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT/set
Mask dir : /kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT/label
Images: 120 Masks: 120 Matched: 120


,split,n
0,train,99
1,test,11
2,val,10


Manifest: /kaggle/working/outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline/02_18_dynunet_phe_only_manifest.csv


,scan_id,case_id,split,image_path,label_path
0,0002,phe_0002,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
1,0029,phe_0029,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
2,0033,phe_0033,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
3,0051,phe_0051,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
4,0061,phe_0061,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...


## 2. Volume IO, metrics, cache

In [4]:
def read_nifti(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return arr, img


def write_nifti_like(arr_zyx, reference_img, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_img = sitk.GetImageFromArray(arr_zyx.astype(np.uint8))
    out_img.CopyInformation(reference_img)
    sitk.WriteImage(out_img, str(out_path))


def normalize_ct(arr, window=CT_WINDOW):
    arr = arr.astype(np.float32)
    lo, hi = window
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / max(hi - lo, 1e-6)
    arr = arr * 2.0 - 1.0
    return arr.astype(np.float32)


def crop_with_pad(arr, center_zyx, patch_size, pad_value=0):
    patch_size = np.asarray(patch_size, dtype=np.int64)
    center = np.asarray(center_zyx, dtype=np.int64)
    start = center - patch_size // 2
    end = start + patch_size

    src_start = np.maximum(start, 0)
    src_end = np.minimum(end, np.asarray(arr.shape, dtype=np.int64))
    dst_start = src_start - start
    dst_end = dst_start + (src_end - src_start)

    out = np.full(tuple(patch_size), pad_value, dtype=arr.dtype)
    out[
        dst_start[0]:dst_end[0],
        dst_start[1]:dst_end[1],
        dst_start[2]:dst_end[2],
    ] = arr[
        src_start[0]:src_end[0],
        src_start[1]:src_end[1],
        src_start[2]:src_end[2],
    ]
    return out


def binary_metrics(pred, ref, eps=1e-8):
    pred = pred.astype(bool)
    ref = ref.astype(bool)
    tp = np.logical_and(pred, ref).sum(dtype=np.float64)
    fp = np.logical_and(pred, ~ref).sum(dtype=np.float64)
    fn = np.logical_and(~pred, ref).sum(dtype=np.float64)
    pred_sum = pred.sum(dtype=np.float64)
    ref_sum = ref.sum(dtype=np.float64)
    dice = (2.0 * tp + eps) / (pred_sum + ref_sum + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return {
        'Dice': float(dice),
        'IoU': float(iou),
        'Precision': float(precision),
        'Recall': float(recall),
        'pred_voxels': int(pred_sum),
        'ref_voxels': int(ref_sum),
    }


class VolumeCache:
    def __init__(self, max_items=MAX_CACHE_VOLUMES):
        self.max_items = int(max_items)
        self._cache = OrderedDict()

    def get(self, row):
        case_id = str(row['case_id'])
        if case_id in self._cache:
            self._cache.move_to_end(case_id)
            return self._cache[case_id]

        image_arr, image_obj = read_nifti(row['image_path'])
        label_arr, label_obj = read_nifti(row['label_path'])
        if image_obj.GetSize() != label_obj.GetSize():
            raise ValueError(f'Size mismatch for {case_id}: image {image_obj.GetSize()} vs label {label_obj.GetSize()}')

        image_arr = normalize_ct(image_arr)
        label_arr = (label_arr > 0).astype(np.uint8)
        item = {
            'case_id': case_id,
            'image': image_arr,
            'label': label_arr,
            'image_obj': image_obj,
            'fg_voxels': np.argwhere(label_arr > 0),
        }
        self._cache[case_id] = item
        self._cache.move_to_end(case_id)
        while len(self._cache) > self.max_items:
            self._cache.popitem(last=False)
        return item

    def clear(self):
        self._cache.clear()
        gc.collect()


train_df = split_df.loc[split_df['split'] == 'train'].reset_index(drop=True)
val_df = split_df.loc[split_df['split'] == 'val'].reset_index(drop=True)
test_df = split_df.loc[split_df['split'] == 'test'].reset_index(drop=True)
print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))


train: 99 val: 10 test: 11


## 3. Dataset and augmentation

In [5]:
AUG_FLIP_PROB = 0.5
AUG_INTENSITY_SCALE = 0.10
AUG_INTENSITY_SHIFT = 0.10
AUG_GAUSSIAN_NOISE_STD = 0.03


def augment_patch(x, y):
    for axis in range(3):
        if random.random() < AUG_FLIP_PROB:
            x = np.flip(x, axis=axis)
            y = np.flip(y, axis=axis)

    if AUG_INTENSITY_SCALE > 0:
        scale = 1.0 + random.uniform(-AUG_INTENSITY_SCALE, AUG_INTENSITY_SCALE)
        x = x * scale
    if AUG_INTENSITY_SHIFT > 0:
        shift = random.uniform(-AUG_INTENSITY_SHIFT, AUG_INTENSITY_SHIFT)
        x = x + shift
    if AUG_GAUSSIAN_NOISE_STD > 0 and random.random() < 0.5:
        x = x + np.random.normal(0.0, AUG_GAUSSIAN_NOISE_STD, size=x.shape).astype(np.float32)

    x = np.clip(x, -1.5, 1.5)
    return np.ascontiguousarray(x), np.ascontiguousarray(y)


class RandomPatchDataset(Dataset):
    def __init__(self, rows_df, cache, patches_per_epoch, patch_size=PATCH_SIZE, fg_patch_prob=FG_PATCH_PROB, augment=True):
        self.rows = rows_df.reset_index(drop=True).to_dict('records')
        self.cache = cache
        self.patches_per_epoch = int(patches_per_epoch)
        self.patch_size = tuple(int(x) for x in patch_size)
        self.fg_patch_prob = float(fg_patch_prob)
        self.augment = bool(augment)
        self._active_row = None
        self._active_left = 0

    def __len__(self):
        return self.patches_per_epoch

    def __getitem__(self, idx):
        if self._active_row is None or self._active_left <= 0:
            self._active_row = self.rows[random.randrange(len(self.rows))]
            self._active_left = PATCHES_PER_CASE_BURST
        row = self._active_row
        self._active_left -= 1
        item = self.cache.get(row)
        image = item['image']
        label = item['label']
        fg_voxels = item['fg_voxels']

        if len(fg_voxels) > 0 and random.random() < self.fg_patch_prob:
            center = fg_voxels[random.randrange(len(fg_voxels))]
            jitter = np.asarray([random.randint(-8, 8), random.randint(-32, 32), random.randint(-32, 32)])
            center = np.clip(center + jitter, 0, np.asarray(image.shape) - 1)
        else:
            center = np.asarray([random.randrange(s) for s in image.shape])

        x = crop_with_pad(image, center, self.patch_size, pad_value=-1.0)
        y = crop_with_pad(label, center, self.patch_size, pad_value=0)
        if self.augment:
            x, y = augment_patch(x, y)

        x = torch.from_numpy(x[None].astype(np.float32))
        y = torch.from_numpy(y[None].astype(np.int64))
        return x, y


train_cache = VolumeCache(MAX_CACHE_VOLUMES)
train_dataset = RandomPatchDataset(train_df, train_cache, TRAIN_PATCHES_PER_EPOCH)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

x0, y0 = train_dataset[0]
print('sample x:', tuple(x0.shape), x0.dtype, float(x0.min()), float(x0.max()))
print('sample y:', tuple(y0.shape), y0.dtype, 'fg voxels:', int((y0 > 0).sum()))


sample x: (1, 32, 128, 128) torch.float32 -1.0585877895355225 1.0211880207061768
sample y: (1, 32, 128, 128) torch.int64 fg voxels: 2038


## 4. Build official MONAI DynUNet

In [6]:
def build_dynunet():
    model = DynUNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        kernel_size=DYNUNET_KERNELS,
        strides=DYNUNET_STRIDES,
        upsample_kernel_size=DYNUNET_UPSAMPLE_KERNELS,
        filters=DYNUNET_FILTERS,
        norm_name=('INSTANCE', {'affine': True}),
        act_name=('leakyrelu', {'inplace': True, 'negative_slope': 0.01}),
        deep_supervision=DYNUNET_DEEP_SUPERVISION,
        deep_supr_num=DYNUNET_DEEP_SUPR_NUM,
        res_block=DYNUNET_RES_BLOCK,
    )
    return model


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_dynunet().to(device)
num_params = sum(p.numel() for p in model.parameters())
print('Device:', device)
print(f'DynUNet params: {num_params / 1e6:.2f}M')
print('kernels:', DYNUNET_KERNELS)
print('strides:', DYNUNET_STRIDES)
print('filters:', DYNUNET_FILTERS)

loss_function = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    include_background=False,
    squared_pred=True,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
scaler = GradScaler(enabled=USE_AMP and device.type == 'cuda')


def compute_loss(outputs, labels):
    if outputs.ndim == 6:
        weights = torch.tensor([1.0 / (2 ** i) for i in range(outputs.shape[1])], device=outputs.device)
        weights = weights / weights.sum()
        total = 0.0
        for i in range(outputs.shape[1]):
            total = total + weights[i] * loss_function(outputs[:, i], labels)
        return total
    return loss_function(outputs, labels)


Device: cuda
DynUNet params: 16.64M
kernels: ((1, 3, 3), (3, 3, 3), (3, 3, 3), (3, 3, 3), (3, 3, 3))
strides: ((1, 1, 1), (1, 2, 2), (2, 2, 2), (2, 2, 2), (2, 2, 2))
filters: (32, 64, 128, 256, 320)


/tmp/ipykernel_23/3185545883.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP and device.type == 'cuda')


## 5. Validation/inference helpers

In [7]:
@torch.no_grad()
def predict_volume(model, image_arr, device):
    model.eval()
    image_norm = normalize_ct(image_arr)
    x = torch.from_numpy(image_norm[None, None].astype(np.float32))
    with autocast(enabled=USE_AMP and device.type == 'cuda'):
        logits = sliding_window_inference(
            x,
            roi_size=PATCH_SIZE,
            sw_batch_size=SLIDING_WINDOW_BATCH_SIZE,
            predictor=model,
            sw_device=device,
            device=torch.device('cpu'),
            overlap=SLIDING_WINDOW_OVERLAP,
            mode='gaussian',
        )
        pred = torch.argmax(logits, dim=1)
    pred_np = pred.squeeze(0).detach().cpu().numpy().astype(np.uint8)
    return pred_np


def validate_model(model, rows_df, device):
    rows = []
    for row in rows_df.itertuples(index=False):
        image_arr, _ = read_nifti(row.image_path)
        label_arr, _ = read_nifti(row.label_path)
        pred = predict_volume(model, image_arr, device)
        metrics = binary_metrics(pred > 0, label_arr > 0)
        metrics.update({'case_id': row.case_id, 'scan_id': row.scan_id})
        rows.append(metrics)
        print(f"val {row.case_id}: Dice={metrics['Dice']:.4f}, IoU={metrics['IoU']:.4f}")
    val_metrics_df = pd.DataFrame(rows)
    mean_dice = float(val_metrics_df['Dice'].mean()) if len(val_metrics_df) else 0.0
    return mean_dice, val_metrics_df


def save_checkpoint(path, epoch, best_val_dice):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': lr_scheduler.state_dict(),
        'best_val_dice': float(best_val_dice),
        'config': {
            'MAX_EPOCHS': MAX_EPOCHS,
            'PATCH_SIZE': PATCH_SIZE,
            'DYNUNET_KERNELS': DYNUNET_KERNELS,
            'DYNUNET_STRIDES': DYNUNET_STRIDES,
            'DYNUNET_FILTERS': DYNUNET_FILTERS,
            'CT_WINDOW': CT_WINDOW,
        },
    }, path)
    print('Saved checkpoint:', path)


def load_checkpoint(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


## 6. Train DynUNet

In [8]:
RUN_TRAIN = True
RESUME_FROM_LATEST = True

latest_ckpt = CHECKPOINT_DIR / 'checkpoint_latest.pth'
best_ckpt = CHECKPOINT_DIR / 'checkpoint_best.pth'
history_csv = OUTPUT_ROOT / '02_18_dynunet_training_history.csv'

start_epoch = 1
best_val_dice = -1.0
history = []

if RESUME_FROM_LATEST and latest_ckpt.exists():
    print('Resuming from:', latest_ckpt)
    ckpt = load_checkpoint(latest_ckpt, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt:
        lr_scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = int(ckpt.get('epoch', 0)) + 1
    best_val_dice = float(ckpt.get('best_val_dice', -1.0))
    if history_csv.exists():
        history = pd.read_csv(history_csv).to_dict('records')
    print('start_epoch:', start_epoch, 'best_val_dice:', best_val_dice)

if RUN_TRAIN:
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        t0 = time.time()
        epoch_losses = []
        for step, (x, y) in enumerate(train_loader, start=1):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=USE_AMP and device.type == 'cuda'):
                outputs = model(x)
                loss = compute_loss(outputs, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            epoch_losses.append(float(loss.detach().cpu()))

        lr_scheduler.step()
        train_loss = float(np.mean(epoch_losses)) if epoch_losses else float('nan')
        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'lr': float(optimizer.param_groups[0]['lr']),
            'epoch_minutes': (time.time() - t0) / 60.0,
        }

        if epoch == 1 or epoch % VALIDATE_EVERY == 0 or epoch == MAX_EPOCHS:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            val_dice, val_metrics_df = validate_model(model, val_df, device)
            row['val_dice'] = val_dice
            val_metrics_df.to_csv(OUTPUT_ROOT / f'02_18_dynunet_val_metrics_epoch_{epoch:03d}.csv', index=False)
            print(f'Epoch {epoch:03d}: train_loss={train_loss:.4f}, val_dice={val_dice:.4f}')
            if val_dice > best_val_dice:
                best_val_dice = val_dice
                save_checkpoint(best_ckpt, epoch, best_val_dice)
                print('New best val Dice:', best_val_dice)
        else:
            row['val_dice'] = np.nan
            print(f'Epoch {epoch:03d}: train_loss={train_loss:.4f}')

        history.append(row)
        pd.DataFrame(history).to_csv(history_csv, index=False)
        if epoch % SAVE_EVERY == 0 or epoch == MAX_EPOCHS:
            save_checkpoint(latest_ckpt, epoch, best_val_dice)

    print('Training done. Best val Dice:', best_val_dice)
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


/tmp/ipykernel_23/1608007895.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP and device.type == 'cuda'):
/tmp/ipykernel_23/3570884981.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP and device.type == 'cuda'):
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/uti

val phe_0013: Dice=0.0000, IoU=0.0000
val phe_0014: Dice=0.0000, IoU=0.0000
val phe_0060: Dice=0.0000, IoU=0.0000
val phe_0078: Dice=0.0000, IoU=0.0000
val phe_0080: Dice=0.0000, IoU=0.0000
val phe_0115: Dice=0.0000, IoU=0.0000
val phe_0119: Dice=0.0000, IoU=0.0000
val phe_0130: Dice=0.0000, IoU=0.0000
val phe_0138: Dice=0.0000, IoU=0.0000
val phe_0160: Dice=0.0000, IoU=0.0000
Epoch 001: train_loss=1.2169, val_dice=0.0000
Saved checkpoint: /kaggle/working/outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline/checkpoints/checkpoint_best.pth
New best val Dice: 5.823062058408703e-12
Epoch 002: train_loss=0.9614
Epoch 003: train_loss=0.8475
Epoch 004: train_loss=0.7767
Epoch 005: train_loss=0.7473
Epoch 006: train_loss=0.7277
Epoch 007: train_loss=0.7367
Epoch 008: train_loss=0.7308
Epoch 009: train_loss=0.6981
val phe_0013: Dice=0.2213, IoU=0.1244
val phe_0014: Dice=0.1833, IoU=0.1009
val phe_0060: Dice=0.2524, IoU=0.1444
val phe_0078: Dice=0.3256, IoU=0.1944
val phe_0080: Dice=0.3689

## 7. Predict and evaluate test set

In [9]:
RUN_TEST = True
CHECKPOINT_TO_USE = 'best'  # 'best' or 'latest'

if CHECKPOINT_TO_USE == 'best':
    ckpt_path = best_ckpt
else:
    ckpt_path = latest_ckpt

if not ckpt_path.exists():
    raise FileNotFoundError(f'Missing checkpoint: {ckpt_path}')

ckpt = load_checkpoint(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
model.eval()
print('Loaded checkpoint:', ckpt_path)
print('checkpoint epoch:', ckpt.get('epoch'), 'best_val_dice:', ckpt.get('best_val_dice'))

if RUN_TEST:
    rows = []
    for row in test_df.itertuples(index=False):
        print('Predicting:', row.case_id)
        image_arr, image_obj = read_nifti(row.image_path)
        label_arr, _ = read_nifti(row.label_path)
        pred = predict_volume(model, image_arr, device)
        pred_path = PRED_DIR / f'{row.case_id}.nii.gz'
        write_nifti_like(pred, image_obj, pred_path)
        metrics = binary_metrics(pred > 0, label_arr > 0)
        metrics.update({
            'scan_id': row.scan_id,
            'case_id': row.case_id,
            'prediction_file': str(pred_path),
            'reference_file': row.label_path,
        })
        rows.append(metrics)
        print(f"done {row.case_id}: Dice={metrics['Dice']:.4f}, IoU={metrics['IoU']:.4f}")

    metrics_df = pd.DataFrame(rows)
    metrics_csv = OUTPUT_ROOT / '02_18_dynunet_phe_only_metrics_per_case.csv'
    metrics_df.to_csv(metrics_csv, index=False)

    summary_rows = []
    for col in ['Dice', 'IoU', 'Precision', 'Recall', 'pred_voxels', 'ref_voxels']:
        vals = pd.to_numeric(metrics_df[col], errors='coerce').dropna()
        summary_rows.append({
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = OUTPUT_ROOT / '02_18_dynunet_phe_only_metrics_summary.csv'
    summary_json = OUTPUT_ROOT / '02_18_dynunet_phe_only_metrics_summary.json'
    summary_df.to_csv(summary_csv, index=False)
    summary_json.write_text(json.dumps({
        'checkpoint': str(ckpt_path),
        'checkpoint_epoch': int(ckpt.get('epoch', -1)),
        'best_val_dice': float(ckpt.get('best_val_dice', np.nan)),
        'test_summary': summary_df.to_dict('records'),
        'test_per_case': metrics_df.to_dict('records'),
    }, indent=2), encoding='utf-8')

    print('Wrote:', metrics_csv)
    print('Wrote:', summary_csv)
    print('Wrote:', summary_json)
    display(summary_df)
else:
    print('Skipped. Set RUN_TEST=True after training.')


Loaded checkpoint: /kaggle/working/outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline/checkpoints/checkpoint_best.pth
checkpoint epoch: 190 best_val_dice: 0.339861612676927
Predicting: phe_0002


/tmp/ipykernel_23/3570884981.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP and device.type == 'cuda'):


done phe_0002: Dice=0.5659, IoU=0.3946
Predicting: phe_0029
done phe_0029: Dice=0.3665, IoU=0.2243
Predicting: phe_0033
done phe_0033: Dice=0.4594, IoU=0.2982
Predicting: phe_0051
done phe_0051: Dice=0.1212, IoU=0.0645
Predicting: phe_0061
done phe_0061: Dice=0.2164, IoU=0.1213
Predicting: phe_0084
done phe_0084: Dice=0.6132, IoU=0.4422
Predicting: phe_0095
done phe_0095: Dice=0.3078, IoU=0.1819
Predicting: phe_0096
done phe_0096: Dice=0.6183, IoU=0.4474
Predicting: phe_0113
done phe_0113: Dice=0.1912, IoU=0.1057
Predicting: phe_0116
done phe_0116: Dice=0.3135, IoU=0.1859
Predicting: phe_0167
done phe_0167: Dice=0.0000, IoU=0.0000
Wrote: /kaggle/working/outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline/02_18_dynunet_phe_only_metrics_per_case.csv
Wrote: /kaggle/working/outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline/02_18_dynunet_phe_only_metrics_summary.csv
Wrote: /kaggle/working/outputs_02_18_kaggle_monai_dynunet_phe_only_fair_baseline/02_18_dynunet_phe_only_metrics_

,label,metric,mean,median,std,n
0,PHE,Dice,0.343030,0.313528,0.195545,11
1,PHE,IoU,0.224191,0.185908,0.146330,11
2,PHE,Precision,0.346687,0.238631,0.258151,11
3,PHE,Recall,0.485346,0.550055,0.258371,11
4,PHE,pred_voxels,3627.545455,2165.000000,4052.594822,11
5,PHE,ref_voxels,2452.000000,1635.000000,2110.821512,11


## Fairness checklist

- Same input as `02_12`: CT only.
- Same target as `02_12`: binary PHE mask.
- Same split IDs: train 99, val 10, test 11.
- Same epoch budget as original `02_12`: 250 epochs.
- Full-volume sliding-window inference on every test case.
- No ICH teacher, no ICH prior, no pseudo ICH channel.

Caveat: `02_12` is nnU-Net `3d_fullres` with nnU-Net preprocessing/trainer. This notebook uses MONAI DynUNet with a custom training loop, so compare as an architecture/framework baseline, not as a controlled trainer-only swap.
